# Step Two: SSL Experiment 1
### for Assignment 2: SSL
**By Tahiyat Chy**

This model is trained using mask autoencoding (MAE).

## Initial Setup

In [1]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import math

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA available: True


In [2]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive
Mounted at /content/gdrive


In [3]:
# Path set to assignment directory

PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169/assignment2')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test
Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test


In [4]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation
Define data augmentations. Split the training images into training and validation set.

In [5]:
import torchvision.transforms as T

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_CLASSES = 16

# Data augmentation
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandAugment(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1)
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_ROOT)

# Split into train and val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Implement custom wrapper to apply different transforms to train and val subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

Total training images: 1920
Total validation images: 480
Total training images: 1920
Total validation images: 480


## Model Architecture
Vision Transformer (ViT-B/16) initialized with weights=None.

In [6]:
class PatchEmbed(nn.Module):
    """2D Image to Patch Embedding"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # [B, embed_dim, H/P, W/P]
        x = x.flatten(2).transpose(1, 2)  # [B, num_patches, embed_dim]
        return x


def patchify(imgs, patch_size=16):
    """
    imgs: (N, 3, H, W)
    x: (N, L, patch_size**2 * 3)
    """
    p = patch_size
    assert imgs.shape[2] == imgs.shape[3] and imgs.shape[2] % p == 0
    h = w = imgs.shape[2] // p
    x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
    x = torch.einsum('nhpwcq->nhwpqc', x)
    x = x.reshape(shape=(imgs.shape[0], h * w, p * p * 3))
    return x


def unpatchify(x, patch_size=16):
    """
    x: (N, L, patch_size**2 * 3)
    imgs: (N, 3, H, W)
    """
    p = patch_size
    h = w = int(x.shape[1] ** 0.5)
    assert h * w == x.shape[1]
    x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
    x = torch.einsum('nhwpqc->nhpwcq', x)
    imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))
    return imgs


class MaskedAutoencoderViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        decoder_embed_dim=512,
        decoder_depth=4,
        decoder_num_heads=16,
        mlp_ratio=4.0,
        norm_pix_loss=True,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.norm_pix_loss = norm_pix_loss

        # --------------------------------------------------------------------------
        # MAE Encoder
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.encoder_norm = nn.LayerNorm(embed_dim)
        # --------------------------------------------------------------------------

        # --------------------------------------------------------------------------
        # MAE Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim))

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_embed_dim,
            nhead=decoder_num_heads,
            dim_feedforward=int(decoder_embed_dim * mlp_ratio),
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * in_chans, bias=True)
        # --------------------------------------------------------------------------

        self._initialize_weights()

    def _initialize_weights(self):
        torch.nn.init.trunc_normal_(self.pos_embed, std=0.02)
        torch.nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.normal_(self.cls_token, std=0.02)
        torch.nn.init.normal_(self.mask_token, std=0.02)

    def random_masking(self, x, mask_ratio=0.75):
        """
        Perform per-sample random masking by shuffling patch indices.
        x: [N, L, D], sequence
        """
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))

        noise = torch.rand(N, L, device=x.device)  # noise in [0, 1]

        # Sort noise for each sample
        ids_shuffle = torch.argsort(noise, dim=1)  # ascend: small is keep, large is remove
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        # Keep first subset
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))

        # Generate binary mask: 0 is keep, 1 is remove
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio=0.75):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]

        # Masking: [N, L, D] -> [N, L*(1-mask_ratio), D]
        x, mask, ids_restore = self.random_masking(x, mask_ratio)

        # Append CLS token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Apply Transformer blocks
        x = self.encoder(x)
        x = self.encoder_norm(x)

        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)

        # Append mask tokens to sequence
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
        x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)  # exclude CLS
        x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))  # unshuffle
        x = torch.cat([x[:, :1, :], x_], dim=1)  # append CLS back

        x = x + self.decoder_pos_embed

        x = self.decoder(x)
        x = self.decoder_norm(x)

        # Predict patch pixels
        x = self.decoder_pred(x)
        x = x[:, 1:, :]  # remove CLS token

        return x

    def forward_loss(self, imgs, pred, mask):
        target = patchify(imgs, self.patch_size)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6) ** 0.5

        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)  # [N, L], loss per patch

        loss = (loss * mask).sum() / mask.sum()  # Mean loss over masked patches only
        return loss

    def forward(self, imgs, mask_ratio=0.75):
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask

## Training Loop
Using CrossEntropyLoss and AdamW optimizer.

In [7]:
def pretrain_mae(model, train_loader, epochs=50, lr=1e-3, mask_ratio=0.75):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05, betas=(0.9, 0.95))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    model.train()
    print("--- Starting MAE Pre-training ---")
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, _ in train_loader:  # Labels are ignored
            inputs = inputs.to(device)

            optimizer.zero_grad()
            loss, _, _ = model(inputs, mask_ratio=mask_ratio)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        scheduler.step()
        epoch_loss = running_loss / len(train_loader.dataset)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Pre-train Epoch [{epoch+1}/{epochs}] | Reconstruct Loss: {epoch_loss:.5f}")

    print("MAE Pre-training Complete.\n")
    return model


# Instantiate and pre-train MAE
mae_model = MaskedAutoencoderViT(img_size=IMG_SIZE, patch_size=16).to(device)
mae_model = pretrain_mae(mae_model, train_loader, epochs=30, lr=1e-3, mask_ratio=0.75)

/tmp/ipykernel_9900/2458296226.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
/tmp/ipykernel_9900/2458296226.py:105: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)


--- Starting MAE Pre-training ---
Pre-train Epoch [1/30] | Reconstruct Loss: 1.04548
Pre-train Epoch [5/30] | Reconstruct Loss: 0.98201
Pre-train Epoch [10/30] | Reconstruct Loss: 0.96131
Pre-train Epoch [15/30] | Reconstruct Loss: 0.95237
Pre-train Epoch [20/30] | Reconstruct Loss: 0.94741
Pre-train Epoch [25/30] | Reconstruct Loss: 0.93945
Pre-train Epoch [30/30] | Reconstruct Loss: 0.94267
MAE Pre-training Complete.



## Model Fine-Tuning

In [8]:
class ViTForClassification(nn.Module):
    def __init__(self, mae_model, num_classes=16):
        super().__init__()
        # Share or copy pre-trained encoder modules
        self.patch_embed = mae_model.patch_embed
        self.cls_token = mae_model.cls_token
        self.pos_embed = mae_model.pos_embed
        self.encoder = mae_model.encoder
        self.encoder_norm = mae_model.encoder_norm

        # Classification head
        self.head = nn.Linear(mae_model.embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]

        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.encoder(x)
        x = self.encoder_norm(x)

        # Extract [CLS] token output
        cls_out = x[:, 0]
        return self.head(cls_out)


# Initialize downstream classifier with pre-trained encoder weights
finetune_model = ViTForClassification(mae_model, num_classes=NUM_CLASSES).to(device)

In [9]:
def train_finetune(model, train_loader, val_loader, epochs=20, lr=1e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    print("--- Starting Downstream Supervised Fine-Tuning ---")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_acc = corrects.double() / len(val_loader.dataset)
        scheduler.step(epoch_acc)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Acc: {epoch_acc:.4f}")

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_wts = copy.deepcopy(model.state_dict())

    print(f"\nFine-Tuning Complete. Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_wts)
    return model


# Run downstream fine-tuning
final_mae_model = train_finetune(finetune_model, train_loader, val_loader, epochs=20, lr=1e-4)

--- Starting Downstream Supervised Fine-Tuning ---
Epoch 1/20 | Train Loss: 2.7941 | Val Acc: 0.1417
Epoch 2/20 | Train Loss: 2.7302 | Val Acc: 0.1354
Epoch 3/20 | Train Loss: 2.6791 | Val Acc: 0.1104
Epoch 4/20 | Train Loss: 2.6356 | Val Acc: 0.1521
Epoch 5/20 | Train Loss: 2.6000 | Val Acc: 0.1417
Epoch 6/20 | Train Loss: 2.5498 | Val Acc: 0.1521
Epoch 7/20 | Train Loss: 2.5340 | Val Acc: 0.2271
Epoch 8/20 | Train Loss: 2.5151 | Val Acc: 0.1917
Epoch 9/20 | Train Loss: 2.4763 | Val Acc: 0.2396
Epoch 10/20 | Train Loss: 2.4770 | Val Acc: 0.2604
Epoch 11/20 | Train Loss: 2.4441 | Val Acc: 0.2604
Epoch 12/20 | Train Loss: 2.4238 | Val Acc: 0.2396
Epoch 13/20 | Train Loss: 2.4184 | Val Acc: 0.2250
Epoch 14/20 | Train Loss: 2.4375 | Val Acc: 0.2375
Epoch 15/20 | Train Loss: 2.3885 | Val Acc: 0.2583
Epoch 16/20 | Train Loss: 2.3822 | Val Acc: 0.2458
Epoch 17/20 | Train Loss: 2.4195 | Val Acc: 0.2792
Epoch 18/20 | Train Loss: 2.3691 | Val Acc: 0.2500
Epoch 19/20 | Train Loss: 2.3753 | Val A